# Checking the dataset


In [1]:
from fairchem.core.datasets import AseDBDataset
from fairchem.core.datasets.atomic_data import atomicdata_list_to_batch

from torch.utils.data import DataLoader
import tqdm
import torch

!nvidia-smi

/home/karella/.conda/envs/hippynn/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Wed Dec 10 10:55:42 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5090        Off |   00000000:01:00.0 Off |                  N/A |
|  0%   34C    P8             16W /  575W |      18MiB /  32607MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
omol4M_training_path = "/home/karella/Projects/hippynn/train_4M"   # e.g. a folder of *.aselmdb
#omol_validation_path = "/home/karella/Projects/hippynn/validation_250k"  # e.g. a folder of *.aselmdb
#omol_test_path = 

train_dataset = AseDBDataset(
    config=dict(
        src=omol4M_training_path,
        a2g_args=dict(
            task_name="omol",   # molecules task (same task UMA uses for molecules)
            # r_energy / r_forces are on by default in fairchem v2 for EF datasets,
            # but you can set them explicitly if you want:
            r_energy=True,
            r_forces=True,
            r_stress=False,
        ),
        # keep_in_memory=True  # only if your subset fits in RAM
    )
)

train_loader = DataLoader(
    train_dataset,
    batch_size=512,
    shuffle=True,
    collate_fn=atomicdata_list_to_batch,  # turns list[AtomicData] → batched AtomicData
)


# Test random access to the dataset
print(f"Dataset has {len(train_loader)} items")
# Calculate
# Go through batches of data in a loop
num_batchs = 100
for idx, batch in enumerate(tqdm.tqdm(train_loader)):
    atom_numbers = batch.atomic_numbers
    pos = batch.pos
    forces = batch.forces
    energy = batch.energy
    if idx >= num_batchs:
        break


Dataset has 7787 items


  1%|▏         | 100/7787 [00:27<35:45,  3.58it/s]


In [1]:
from hippynn.graphs import inputs, networks, targets, physics
from hippynn.graphs.nodes.loss import MAELoss
from hippynn.experiment import assemble_for_training, lightning_trainer, SetupParams, setup_training, HippynnLightningModule
from hippynn.pretraining import hierarchical_energy_initialization
import lightning as L

# Setting up the network 
network_params = {
    "possible_species": list(range(90)),
    "n_features": 128,
    "n_sensitivities": 20,
    "dist_soft_min": 0.8,
    "dist_soft_max": 5.5,
    "dist_hard_max": 6.5,
    "n_interaction_layers": 2,
    "n_atom_layers": 5,
    "n_max":3,
    "l_max":2,
    "sensitivity_type": "inverse",
    "resnet": True,
}

experiment_params = SetupParams(
    stopping_key="T-MAE",  # The name in the validation_losses dictionary.
    batch_size=12,
    optimizer=torch.optim.Adam,
    max_epochs=100,
    learning_rate=0.001,
)


species = inputs.SpeciesNode(db_name="atomic_numbers")
positions = inputs.PositionsNode(db_name="pos")
#cell = inputs.CellNode(db_name="cell")

network = networks.HipHopnn(name="HipHopnn",
                            parents=(species, positions),
                            module_kwargs=network_params,
                            periodic=False)

henergy = targets.HEnergyNode("HEnergy", network)
sys_energy = henergy.mol_energy
sys_energy.db_name = "energy"
hierarchicality = henergy.hierarchicality
hierarchicality = physics.PerAtom("RperAtom", hierarchicality)
force = physics.GradientNode("forces", (sys_energy, positions), sign=-1)
force.db_name = "forces"

tmae = MAELoss.of_node(sys_energy)

validation_losses = {
"T-MAE": tmae,
"F-MAE":MAELoss.of_node(force),
}
validation_losses['train'] =  validation_losses['T-MAE'] + validation_losses['F-MAE']

train_loss = validation_losses['train']


# NOTE: This part is taken from Nick lightning example
# This piece of code glues the stuff together as a pytorch model,
# dropping things that are irrelevant for the losses defined.
training_modules, db_info = assemble_for_training(tmae, validation_losses)
# Now that we have a database and a model, we can
# Fit the non-interacting energies by examining the database.
# This tends to stabilize training a lot.
# TODO: Fix this
# hierarchical_energy_initialization(sys_energy) 
(model, loss, evaluator), controller, metric_tracker = setup_training(training_modules,
                                                                       experiment_params)

l_module = HippynnLightningModule(
            model=model,
            loss=loss,
            eval_loss=evaluator.loss,
            eval_names=evaluator.loss_names,
            optimizer_list=[controller.optimizer],
            scheduler_list=controller.scheduler_list,
            stopping_key=controller.stopping_key,
            controller=controller,
            metric_tracker=metric_tracker,
            inputs=['atomic_numbers', 'pos'], 
            targets=['energy', 'forces'],
            n_outputs=evaluator.n_outputs
        )
#trainer = L.Trainer(accelerator="gpu", devices=2, max_epochs=experiment_params.max_epochs)
l_module 

/home/karella/.conda/envs/hippynn/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'torch' is not defined

In [11]:


### Read in the dataset you wish to submit predictions to
from hippynn.interfaces.ase_interface.ase_database import AseDatabaseIterable

# Hyperparameters for the network

species = inputs.SpeciesNode(db_name="numbers")
positions = inputs.PositionsNode(db_name="positions")
cell = inputs.CellNode(db_name="cell")

network = networks.HipHopnn("HipHopnn", (species, positions), module_kwargs=network_params, periodic=False)

henergy = targets.HEnergyNode("HEnergy", network)
sys_energy = henergy.mol_energy
sys_energy.db_name = "energy"
hierarchicality = henergy.hierarchicality
hierarchicality = physics.PerAtom("RperAtom", hierarchicality)
force = physics.GradientNode("forces", (sys_energy, positions), sign=-1)
force.db_name = "forces"




validation_losses = {
"T-MAE":MAELoss.of_node(sys_energy),
"F-MAE":MAELoss.of_node(force),
}
validation_losses['train'] =  validation_losses['T-MAE'] + validation_losses['F-MAE']

train_loss = validation_losses['train']


# Factors of 1e3 for meV

training_modules, db_info = assemble_for_training(train_loss, validation_losses)


def stream_conversion_generator(fairchem_db):

    for atom_data in fairchem_db:
        atoms = atom_data.to_ase()[0]
        arrays = atoms.arrays
        #arrays['atomic_numbers'] = atom_data.atomic_numbers.numpy()
        arrays['forces'] = atom_data.forces.numpy()
        arrays['pos'] = atom_data.pos.numpy()
        
        atoms.info['energy'] = atom_data.energy.numpy()
        #atoms.info['charge'] = atom_data.charge
        #atoms.info['spin'] = atom_data.spin
        yield atoms

    return



dataset = AseDBDataset({"src": "/home/karella/Projects/hippynn/train_4M"})  

gen = stream_conversion_generator(dataset[0:10000])

database = AseDatabaseIterable(
    iterable=gen,
    seed=1001,  # Random seed for splitting data
    quiet=False,
    pin_memory=False,
    test_size=0.1,
    valid_size=0.1, 
    **db_info) 

/home/karella/Projects/hippynn/hippynn/networks/hiphop.py:12: UserWarning: HIP-HOP-NN is still in a beta state: Details, defaults, and API are still subject to change.
  warnings.warn("HIP-HOP-NN is still in a beta state: " "Details, defaults, and API are still subject to change.")


Determined Inputs: ["Species(db_name='numbers')", "Positions(db_name='positions')"]
Determined Outputs: ['HEnergy.mol_energy', 'forces']
Determined Targets: ["Species(db_name='numbers')", 'HEnergy.mol_energy', 'forces']
Device was not specified. Attempting to default to device: cuda:0


Data types:
{'numbers': dtype('int64'), 'positions': dtype('float32'), 'forces': dtype('float32'), 'energy': dtype('float32')}
All arrays:
--------------------------------------------------------------------------------------
| Name                           | dtype              | shape                        |
--------------------------------------------------------------------------------------
| numbers                        | dtype('int64')     | (10000, 344)                 |
| positions                      | dtype('float32')   | (10000, 344, 3)              |
| forces                         | dtype('float32')   | (10000, 344, 3)              |
| energy                         | dtype('float32')   | (10000, 1)                   |
--------------------------------------------------------------------------------------
Finished checking input and target arrays; all necessary arrays were found.
Database: Using auto-generated data indices
Adding split indices for split: test
Arrays f

In [ ]:
database.

['numbers', 'energy', 'forces']